In [1]:
!pip install tensorflow-model-optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 6.4 MB/s eta 0:00:00


In [3]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
import tensorflow_model_optimization as tfmot

# --- Pre-requisite: Generate Baseline Model ---
# Creating the model locally since Colab will not have the file from "Experiment 2"
os.makedirs('models', exist_ok=True)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.astype('float32').reshape(-1, 784) / 255.0
X_test = X_test.astype('float32').reshape(-1, 784) / 255.0

print("Generating baseline model...")
temp_model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(784,)),
    keras.layers.Dense(10, activation='softmax')
])
temp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
temp_model.fit(X_train, y_train, epochs=3, verbose=0)
temp_model.save('models/baseline_ann.h5')
print("Baseline model saved.\n")


# --- Lab 7 Implementation Starts Here ---
# Step 1: Load Previously Trained Model
# Corrected missing '=' operator
model = keras.models.load_model('models/baseline_ann.h5')

# Step 2: Baseline Measurement
def get_size(path):
    return os.path.getsize(path) / 1024

def measure_latency(model_fn, data, runs=100):
    start = time.time()
    # Corrected missing variable in loop
    for _ in range(runs):
        model_fn(data[:1])
    # Corrected missing '-' and '*' operators
    return (time.time() - start) / runs * 1000 # ms

baseline_size = get_size('models/baseline_ann.h5')
baseline_loss, baseline_acc = model.evaluate(X_test, y_test, verbose=0)
baseline_latency = measure_latency(model.predict, X_test)

print(f'Baseline size (KB): {baseline_size:.1f}')
print(f'Baseline accuracy: {baseline_acc:.4f}')
print(f'Baseline latency (ms): {baseline_latency:.2f}\n')

# Step 3: Post-Training Quantization
# Corrected missing '=' operators
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_tflite_model = converter.convert()

with open('models/quantized_model.tflite', 'wb') as f:
    f.write(quantized_tflite_model)

# Corrected missing '=' operator
quant_size = get_size('models/quantized_model.tflite')
print(f'Quantized size (KB): {quant_size:.1f}\n')

# Step 4: Magnitude-Based Pruning
# Corrected missing '=' operator
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=1000
    )
}

pruned_model = prune_low_magnitude(model, **pruning_params)
pruned_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
# Warning: Fine-tuning on X_test/y_test causes data leakage. Executing as written in the lab manual.
pruned_model.fit(X_test, y_test, epochs=2, callbacks=callbacks, verbose=0)

# Step 5: Strip pruning wrappers and convert the pruned model
final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# FIX: Re-compile the stripped model before evaluation or saving
final_pruned_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

final_pruned_model.save('models/pruned_model.h5')

pruned_size = get_size('models/pruned_model.h5')
pruned_loss, pruned_acc = final_pruned_model.evaluate(X_test, y_test, verbose=0)

print(f'Pruned size (KB): {pruned_size:.1f}')
print(f'Pruned accuracy: {pruned_acc:.4f}')

Generating baseline model...
Baseline model saved.

1/1 [==============================] - 0s 20ms/step
Baseline size (KB): 1217.0
Baseline accuracy: 0.9738
Baseline latency (ms): 63.68

Quantized size (KB): 103.1



/usr/local/lib/python3.13/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Pruned size (KB): 413.4
Pruned accuracy: 0.0975
